# Fresnel reflection at a dielectric interface

This notebook accompanies the **Photonic Devices 2026** course.

We consider a planar interface between two lossless, non-magnetic dielectric media with refractive indices $n_1$ and $n_2$.

The notebook explores:
- TE and TM Fresnel reflection coefficients;
- reflectivity versus incidence angle;
- Brewster angle;
- critical angle and total internal reflection;
- reflection phase in the TIR regime.


## 1. Parameters

Angles are measured from the interface normal.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Refractive indices
n1 = 1.50
n2 = 1.00

# Incidence-angle grid
theta1_deg = np.linspace(0.0, 89.9, 2000)
theta1 = np.deg2rad(theta1_deg)


## 2. Snell's law

Tangential phase matching gives

$$n_1\sin\theta_1=n_2\sin\theta_2.$$

For $n_1>n_2$, the transmitted angle becomes complex above the critical angle.


In [ ]:
# Snell's law
sin_theta2 = (n1 / n2) * np.sin(theta1)
cos_theta1 = np.cos(theta1)

# Choose the branch consistent with the course convention
# exp(+j omega t) exp(-j k.r): in TIR, k_2z = -j gamma
cos_theta2 = np.empty_like(sin_theta2, dtype=complex)
propagating = sin_theta2 <= 1.0
cos_theta2[propagating] = np.sqrt(1 - sin_theta2[propagating]**2)
cos_theta2[~propagating] = -1j*np.sqrt(sin_theta2[~propagating]**2 - 1)

theta_c = np.arcsin(n2 / n1) if n1 > n2 else None
theta_B = np.arctan(n2 / n1)

print(f"Brewster angle: {np.rad2deg(theta_B):.2f} deg")
if theta_c is not None:
    print(f"Critical angle: {np.rad2deg(theta_c):.2f} deg")


## 3. Fresnel reflection coefficients

For TE polarization,

$$r_{\rm TE}=\frac{n_1\cos\theta_1-n_2\cos\theta_2}{n_1\cos\theta_1+n_2\cos\theta_2}.$$

For TM polarization, using the convention adopted in the course,

$$r_{\rm TM}=\frac{n_2\cos\theta_1-n_1\cos\theta_2}{n_2\cos\theta_1+n_1\cos\theta_2}.$$


In [ ]:
r_TE = (n1*cos_theta1 - n2*cos_theta2) / (n1*cos_theta1 + n2*cos_theta2)
r_TM = (n2*cos_theta1 - n1*cos_theta2) / (n2*cos_theta1 + n1*cos_theta2)

R_TE = np.abs(r_TE)**2
R_TM = np.abs(r_TM)**2


## 4. Reflectivity versus incidence angle


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(theta1_deg, R_TE, label='TE')
plt.plot(theta1_deg, R_TM, label='TM')

plt.axvline(np.rad2deg(theta_B), linestyle='--', label='Brewster angle')
if theta_c is not None:
    plt.axvline(np.rad2deg(theta_c), linestyle=':', label='Critical angle')

plt.xlabel(r'Incidence angle $\theta_1$ (deg)')
plt.ylabel('Reflectivity R')
plt.ylim(0, 1.05)
plt.grid(True)
plt.legend()
plt.show()


### Observations

- At normal incidence, TE and TM have the same reflectivity.
- For oblique incidence, their reflectivities differ.
- The TM reflectivity vanishes at the Brewster angle.
- For $\theta_1>\theta_c$, both polarizations undergo total internal reflection and $R=1$.


## 5. Reflection phase in total internal reflection

Although $|r|=1$ in TIR, the reflection coefficient is generally complex. TE and TM therefore acquire different reflection phases.


In [ ]:
phase_TE = np.angle(r_TE)
phase_TM = np.angle(r_TM)

if theta_c is not None:
    tir = theta1 > theta_c

    plt.figure(figsize=(8,5))
    plt.plot(theta1_deg[tir], phase_TE[tir], label='TE')
    plt.plot(theta1_deg[tir], phase_TM[tir], label='TM')
    plt.axvline(np.rad2deg(theta_c), linestyle=':', label='Critical angle')

    plt.xlabel(r'Incidence angle $\theta_1$ (deg)')
    plt.ylabel('Reflection phase (rad)')
    plt.grid(True)
    plt.legend()
    plt.show()


## 6. Try it yourself

1. Change $n_1$ and $n_2$ and predict how the Brewster angle changes.
2. Verify numerically that

$$\tan\theta_B=\frac{n_2}{n_1}.$$

3. For $n_1>n_2$, verify that

$$\sin\theta_c=\frac{n_2}{n_1}.$$

4. Compare the TE and TM reflection phases in the TIR regime.
5. Try $n_1<n_2$ and observe what changes.
